# M4S3 - HPC and Machine Learning
## Training is not only an ML problem - it is a systems problem

In this notebook we will measure how data loading, batch size, precision, memory and parallelism affect machine-learning training.

**Predict → run → observe → explain**

Run the cells in order. Some later cells check whether PyTorch or TensorFlow is available in the current Jupyter kernel.


## 0 - Check the environment
**Predict:** CPU, memory, storage or GPU - what will limit this run?


In [ ]:
import os, sys, shutil, subprocess, time, importlib.util
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from IPython.display import display, HTML

def sh(cmd):
    p=subprocess.run(cmd,shell=True,text=True,capture_output=True)
    return (p.stdout+p.stderr).strip()

print("host:", sh("hostname"))
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("CPU cores:", os.cpu_count())
print("Slurm:", bool(shutil.which("sbatch")))
print("nvidia-smi:", shutil.which("nvidia-smi"))

if shutil.which("nvidia-smi"):
    print(sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader | head -4"))

cmd='bash -lc \'if type module >/dev/null 2>&1; then module -t avail 2>&1 | grep -Ei "pytorch|tensorflow|cuda|cudnn|nccl" | head -20; else echo "module command not visible"; fi\''
print("\nRelevant modules:\n"+sh(cmd))

def show_bars(title, items, unit="", digits=1):
    vals=[float(v) for _,v in items]
    m=max(vals) if vals else 1.0
    html=f"<div style='font-family:Arial,sans-serif;max-width:820px'><h4>{title}</h4>"
    for label,val in items:
        width=0 if m==0 else 100*float(val)/m
        value=f"{float(val):,.{digits}f}{unit}"
        html += (
            "<div style='display:grid;grid-template-columns:150px 1fr 170px;gap:10px;"
            "align-items:center;margin:5px 0'>"
            f"<div>{label}</div>"
            "<div style='background:#e8edf3;height:18px;border-radius:2px'>"
            f"<div style='width:{width:.1f}%;background:#55758f;height:18px;border-radius:2px'></div></div>"
            f"<div style='text-align:right'>{value}</div></div>"
        )
    display(HTML(html+"</div>"))


## 1 - A tiny ML workload
We use a small softmax classifier. The model is intentionally simple: today we care about **data movement, batching, memory and throughput**, not network architecture.


In [ ]:
rng=np.random.default_rng(42); N,D,C=12000,96,5
X=rng.normal(size=(N,D)).astype(np.float32); wt=rng.normal(size=(D,C)).astype(np.float32)
y=np.argmax(X@wt+0.7*rng.normal(size=(N,C)),axis=1).astype(np.int64)
cut=9600; Xtr,Xv=X[:cut],X[cut:]; ytr,yv=y[:cut],y[cut:]
def softmax(z):
    z=z-z.max(1,keepdims=True); e=np.exp(z); return e/e.sum(1,keepdims=True)
def acc(X,y,W,b): return float(np.mean(np.argmax(X@W+b,1)==y))
def grad(X,y,W,b):
    p=softmax(X@W+b); n=len(X); loss=-np.log(p[np.arange(n),y]+1e-12).mean()
    p[np.arange(n),y]-=1; p/=n; return float(loss),X.T@p,p.sum(0)
def train(bs=128,epochs=4,lr=.18,seed=0):
    rg=np.random.default_rng(seed); W=np.zeros((D,C),np.float32); b=np.zeros(C,np.float32); updates=0; t=time.perf_counter()
    for _ in range(epochs):
        ids=rg.permutation(len(Xtr))
        for s in range(0,len(Xtr),bs):
            q=ids[s:s+bs]; _,gW,gb=grad(Xtr[q],ytr[q],W,b); W-=lr*gW; b-=lr*gb; updates+=1
    return W,b,time.perf_counter()-t,updates
W,b,t,u=train()
print(f"samples={len(Xtr):,}, features={D}, data={Xtr.nbytes/2**20:.2f} MiB")
print(f"time={t:.3f}s, updates={u}, throughput={len(Xtr)*4/t:,.0f} samples/s, val_acc={acc(Xv,yv,W,b):.3f}")


## 2 - [DEMO] Optimized libraries
**Predict:** how much faster is NumPy matrix multiplication than nested Python loops?


In [ ]:
A=rng.normal(size=(1200,256)); v=rng.normal(size=256)
def py_mv(A,v):
    out=np.empty(len(A))
    for i,row in enumerate(A):
        s=0.
        for j,x in enumerate(row): s+=x*v[j]
        out[i]=s
    return out
t=time.perf_counter(); a=py_mv(A,v); tp=time.perf_counter()-t
t=time.perf_counter(); 
for _ in range(30): b=A@v
tn=(time.perf_counter()-t)/30
print(f"Python loops: {tp:.4f}s | NumPy: {tn:.6f}s | speedup: {tp/tn:,.0f}x | same result: {np.allclose(a,b)}")


**Lesson:** PyTorch/TensorFlow Python code often calls highly optimized compiled kernels underneath.


## 3 - [DEMO] Preprocessing can starve the accelerator


In [ ]:
raw=rng.integers(0,256,size=(900,96),dtype=np.uint8)
def slow(a):
    z=np.empty(a.shape,np.float32)
    for i in range(len(a)):
        for j in range(a.shape[1]): z[i,j]=(float(a[i,j])-127.5)/127.5
    return z
def fast(a): return (a.astype(np.float32)-127.5)/127.5
t=time.perf_counter(); z1=slow(raw); ts=time.perf_counter()-t
t=time.perf_counter(); z2=fast(raw); tf=time.perf_counter()-t
print(f"slow={len(raw)/ts:,.0f} samples/s | vectorized={len(raw)/tf:,.0f} samples/s | speedup={ts/tf:.1f}x")
for prep in (30,12,4):
    compute=12; print(f"prep={prep:2} ms, compute=12 ms -> approx accelerator utilization={100*compute/(prep+compute):.1f}%")


**Fix the feeding pipeline before buying more GPUs.**


## 4 - [DEMO] Batch size: throughput ↔ memory ↔ convergence
Train the same model for the same 4 epochs.


In [ ]:
rows=[]
for bs in (16,64,256,1024):
    W,b,t,u=train(bs=bs,epochs=4,seed=1)
    rows.append((bs,u,t,len(Xtr)*4/t,acc(Xv,yv,W,b),bs*D*4/1024))
print(f"{'batch':>6} {'updates':>8} {'time':>8} {'samples/s':>12} {'val_acc':>8} {'batch_KiB':>10}")
for r in rows: print(f"{r[0]:6d} {r[1]:8d} {r[2]:8.3f} {r[3]:12,.0f} {r[4]:8.3f} {r[5]:10.1f}")


In [ ]:
show_bars(
    "Measured throughput by batch size",
    [(f"batch {r[0]}", r[3]) for r in rows],
    unit=" samples/s",
    digits=0
)
show_bars(
    "Approximate input-batch memory",
    [(f"batch {r[0]}", r[5]) for r in rows],
    unit=" KiB",
    digits=1
)
print("Validation accuracy:", {r[0]: round(r[4],3) for r in rows})
print("Question: which batch is best if throughput and validation accuracy both matter?")


Discuss:
- larger batches usually improve hardware utilization;
- memory use grows;
- updates per epoch fall;
- convergence can change;
- the batch with the highest throughput is not necessarily the batch with the best validation result.

**Batch size is both an ML parameter and a systems parameter.**


## 5 - [DEMO] Prefetch hides I/O behind compute


In [ ]:
def load(i): time.sleep(.018); return i
def compute(x): time.sleep(.030)
n=12; t=time.perf_counter()
for i in range(n): compute(load(i))
seq=time.perf_counter()-t
t=time.perf_counter()
with ThreadPoolExecutor(max_workers=1) as ex:
    f=ex.submit(load,0)
    for i in range(n):
        x=f.result()
        if i+1<n: f=ex.submit(load,i+1)
        compute(x)
pre=time.perf_counter()-t
print(f"sequential={seq:.3f}s | prefetched={pre:.3f}s | speedup={seq/pre:.2f}x")


Prefetching does not make I/O or compute individually faster; it improves **pipeline utilization** by overlapping them.


## 6 - [DEMO] Precision changes memory and numerical behaviour


In [ ]:
base=rng.normal(size=(2048,512))
for name,dtype in (("FP64",np.float64),("FP32",np.float32),("FP16",np.float16)):
    a=base.astype(dtype); print(f"{name}: {a.nbytes/2**20:.2f} MiB")
A=base[:512,:256]; B=base[:256,:256]; ref=A@B
for name,dtype in (("FP32",np.float32),("FP16",np.float16)):
    a=A.astype(dtype); b=B.astype(dtype); t=time.perf_counter(); out=a@b; dt=time.perf_counter()-t
    err=np.linalg.norm(out.astype(np.float64)-ref)/np.linalg.norm(ref)
    print(f"{name}: time={dt:.4f}s, relative_difference={err:.3e}")


In [ ]:
precision_memory=[("FP64",8.0),("FP32",4.0),("FP16",2.0)]
show_bars("Memory footprint for the same array", precision_memory, unit=" MiB", digits=1)
print("Important: lower precision reduces memory, but speed depends on hardware support.")


> On CPUs FP16 may not be faster. GPUs can have specialized tensor hardware, so **hardware support matters**.


## 7 - [DEMO] How much memory does training state need?

A model needs more memory for training than for storing its parameters alone.

For a simplified FP32 model trained with Adam:

- parameters: **4 bytes / parameter**
- gradients: **4 bytes / parameter**
- Adam first moment: **4 bytes / parameter**
- Adam second moment: **4 bytes / parameter**

Simplified training-state estimate:

**about 16 bytes per parameter**

This excludes activations, temporary buffers, framework overhead and other copies.

**Predict:** could a 7-billion-parameter model train in this simplified form on one 24 GB GPU?


In [ ]:
models=[
    ("1 million",1_000_000),
    ("100 million",100_000_000),
    ("1 billion",1_000_000_000),
    ("7 billion",7_000_000_000),
]
bytes_per_parameter=16
print(f"{'model size':>14} {'training state':>18}")
memory_rows=[]
for label,nparams in models:
    gb=nparams*bytes_per_parameter/1e9
    memory_rows.append((label,gb))
    print(f"{label:>14} {gb:15.3f} GB")
show_bars("Simplified FP32 + Adam training-state memory", memory_rows, unit=" GB", digits=1)
print("\nA 7B model needs about 112 GB for this simplified training state alone.")
print("Activations and temporary tensors would require additional memory.")


**Key idea:** model size can become an HPC memory problem before compute becomes the main problem.

Mixed precision, sharding and distributed strategies can reduce or distribute parts of this footprint. The distributed strategies belong to the next session.


## 8 - [DEMO] Gradient accumulation: a larger effective batch without a larger physical batch

Suppose GPU memory only allows a physical batch of **64**.

Instead of updating after every batch:

**64 examples → backward → update**

we can accumulate gradients:

**64 + 64 + 64 + 64 → backward four times → one update**

Effective batch size:

**64 × 4 = 256**

This helps with memory limits, but it does **not** provide the same compute speedup as four GPUs working in parallel.


In [ ]:
physical_batch=64
accumulation_steps=4
effective_batch=physical_batch*accumulation_steps
print("physical batch:", physical_batch)
print("accumulation steps:", accumulation_steps)
print("effective batch:", effective_batch)
print("Key point: memory is similar to batch 64, but the optimizer updates every 256 examples.")


**Question:** Is gradient accumulation the same as distributed training?

No. Gradient accumulation changes when the model is updated and helps create a larger effective batch. It does not add more GPUs or run the computation in parallel.


## 9 - [DEMO] Checkpointing is resilience + I/O


In [ ]:
out=Path("m4s3_outputs"); out.mkdir(exist_ok=True); f=out/"checkpoint.npz"
W,b,_,_=train(bs=128,epochs=2,seed=3); before=acc(Xv,yv,W,b)
t=time.perf_counter(); np.savez(f,W=W,b=b,epoch=np.array([2])); save=time.perf_counter()-t
del W,b; t=time.perf_counter(); s=np.load(f); load=time.perf_counter()-t
after=acc(Xv,yv,s["W"],s["b"])
print(f"file={f} | size={f.stat().st_size/1024:.1f} KiB | save={save*1e3:.2f} ms | load={load*1e3:.2f} ms")
print(f"accuracy before={before:.4f}, after reload={after:.4f}, recovered={np.isclose(before,after)}")


At scale, checkpoint frequency trades **recovery time** against **storage/I/O overhead**.

Real training jobs may run for hours or days, so losing all progress after a failure can be very expensive.


## 10 - [DEMO] The same training loop in PyTorch

PyTorch expresses the same training steps we have already used:

**batch → forward pass → loss → backward pass → optimizer update**

We will first check whether PyTorch is available in this Jupyter kernel.


In [ ]:
torch_available = importlib.util.find_spec("torch") is not None
print("PyTorch available in this kernel:", torch_available)

if torch_available:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset

    print("PyTorch version:", torch.__version__)
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)
    if device.type=="cuda":
        print("GPU:", torch.cuda.get_device_name(0))

    torch.manual_seed(7)
    N,D,C=6000,96,5
    X_t=torch.randn(N,D)
    true_w=torch.randn(D,C)
    y_t=(X_t@true_w).argmax(dim=1)

    loader=DataLoader(TensorDataset(X_t,y_t),batch_size=256,shuffle=True,num_workers=0)
    model=nn.Sequential(nn.Linear(D,128),nn.ReLU(),nn.Linear(128,C)).to(device)
    optimizer=torch.optim.SGD(model.parameters(),lr=0.08)
    loss_fn=nn.CrossEntropyLoss()

    t0=time.perf_counter()
    seen=0
    for xb,yb in loader:
        xb=xb.to(device)
        yb=yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss=loss_fn(model(xb),yb)
        loss.backward()
        optimizer.step()
        seen += xb.shape[0]

    if device.type=="cuda":
        torch.cuda.synchronize()
    dt=time.perf_counter()-t0
    print(f"time={dt:.3f}s | throughput={seen/dt:,.0f} examples/s | final loss={float(loss):.4f}")
else:
    print("PyTorch is not part of this Jupyter kernel.")
    print("The code pattern below is the PyTorch equivalent of the training loop used earlier.")


### PyTorch training pattern

```python
model = model.to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.08)

for xb, yb in loader:
    xb, yb = xb.to(device), yb.to(device)

    optimizer.zero_grad()
    prediction = model(xb)
    loss = loss_fn(prediction, yb)

    loss.backward()
    optimizer.step()
```

**Question:** If the GPU is waiting for the DataLoader, will a faster GPU solve the problem?


## 11 - [DEMO] Data parallelism: split the batch, then combine gradients

Start with one batch of **512 examples**.

Split it into four equal shards:

**128 + 128 + 128 + 128**

Each worker computes a local gradient. The gradients are then averaged before the model update.


In [ ]:
ids=rng.choice(len(Xtr),512,replace=False); xb,yb=Xtr[ids],ytr[ids]
W=np.zeros((D,C),np.float32); b=np.zeros(C,np.float32)
_,gf,bf=grad(xb,yb,W,b); gs=[]; bs=[]
for q in np.array_split(np.arange(512),4):
    _,g,bb=grad(xb[q],yb[q],W,b); gs.append(g); bs.append(bb)
ga=np.mean(gs,0); ba=np.mean(bs,0)
print("4 workers x 128 samples")
print("max gradient difference:",float(np.max(np.abs(gf-ga))))
print("max bias-gradient difference:",float(np.max(np.abs(bf-ba))))


The averaged gradient is almost identical to the gradient computed from the full batch.

This is the key idea behind synchronous data parallel training:

**split data → compute locally → reduce gradients → update the same model**

With real GPUs, the gradient reduction becomes communication between devices.


## 12 - PyTorch DistributedDataParallel

PyTorch **DistributedDataParallel (DDP)** normally uses one process per GPU.

With four GPUs:

**GPU 0 ↔ process 0**  
**GPU 1 ↔ process 1**  
**GPU 2 ↔ process 2**  
**GPU 3 ↔ process 3**

Each process has a model replica and receives a different data shard. During the backward pass, DDP synchronizes gradients before the optimizer update.

### Minimal DDP structure

```python
import os
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

dist.init_process_group(backend="nccl")

local_rank = int(os.environ["LOCAL_RANK"])
torch.cuda.set_device(local_rank)

model = MyModel().to(local_rank)
model = DDP(model, device_ids=[local_rank])

for xb, yb in loader:
    xb, yb = xb.to(local_rank), yb.to(local_rank)

    optimizer.zero_grad()
    loss = loss_fn(model(xb), yb)
    loss.backward()       # DDP synchronizes gradients here
    optimizer.step()
```

Launch on one node with four GPUs:

```bash
torchrun --standalone --nproc_per_node=4 train_ddp.py
```


In [ ]:
world_size=4
global_batch=512
local_batch=global_batch//world_size

print("PyTorch DDP example")
print("world size:", world_size)
print("global batch:", global_batch)
print("batch per process/GPU:", local_batch)

for rank in range(world_size):
    print(f"rank {rank}: model replica + {local_batch} examples")

print("\nAfter backward:")
print("local gradients -> collective reduction -> synchronized gradients -> optimizer update")


**What changed compared with one GPU?**

We gained parallel compute, but introduced:

- gradient communication;
- synchronization;
- several processes;
- possible load imbalance.

More GPUs help only when the extra computation is large enough to justify these costs.


## 13 - TensorFlow MirroredStrategy

TensorFlow **MirroredStrategy** implements synchronous data parallelism on multiple devices in one machine.

With four GPUs:

1. the model is replicated on all four GPUs;
2. the global batch is split into four local batches;
3. each GPU computes local gradients;
4. gradients are reduced across replicas;
5. all replicas receive the same updated weights.

The systems pattern is the same as PyTorch DDP.


In [ ]:
tf_available = importlib.util.find_spec("tensorflow") is not None
print("TensorFlow available in this kernel:", tf_available)

if tf_available:
    import tensorflow as tf
    print("TensorFlow version:", tf.__version__)
    print("Visible GPUs:", [d.name for d in tf.config.list_physical_devices("GPU")])

    if tf.config.list_physical_devices("GPU"):
        strategy=tf.distribute.MirroredStrategy()
        print("MirroredStrategy replicas:", strategy.num_replicas_in_sync)
    else:
        print("No GPU is allocated to this Jupyter kernel.")
else:
    print("TensorFlow is not part of this Jupyter kernel.")

print("\nMirroredStrategy pattern:")
print("replicate model -> split global batch -> local gradients -> reduce -> synchronized update")


### TensorFlow training pattern

```python
strategy = tf.distribute.MirroredStrategy()

with strategy.scope():
    model = build_model()
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy"
    )

model.fit(dataset)
```

### Across several nodes

TensorFlow uses **MultiWorkerMirroredStrategy**:

```python
strategy = tf.distribute.MultiWorkerMirroredStrategy()
```

The workers must know the cluster membership, usually through `TF_CONFIG`.


## 14 - Which distributed strategy fits which problem?

| Problem | Strategy | Example |
|---|---|---|
| Model fits on one GPU, training is slow | **Data parallelism** | ResNet training on 4 GPUs |
| One node is not enough | **Multi-worker data parallelism** | 2 nodes × 4 GPUs |
| Training state is too large for one GPU | **State sharding** | PyTorch FSDP |
| Model layers do not fit on one GPU | **Model / pipeline parallelism** | very large transformer |

### Framework names

**PyTorch:** DDP for data parallelism, FSDP for sharded training state  
**TensorFlow:** MirroredStrategy on one host, MultiWorkerMirroredStrategy across hosts


In [ ]:
examples=[
    ("ResNet fits on every GPU, but training is slow", "Data parallelism"),
    ("8 GPUs across 2 nodes", "Multi-worker data parallelism"),
    ("Training state is larger than one GPU memory", "State sharding"),
    ("The model itself does not fit on one GPU", "Model or pipeline parallelism")
]
for problem,strategy in examples:
    print(f"{problem}\n  -> {strategy}\n")


### Worked example: 4 GPUs with data parallelism

**Global batch = 512**

- GPU 0 gets 128 examples
- GPU 1 gets 128 examples
- GPU 2 gets 128 examples
- GPU 3 gets 128 examples

All four GPUs hold the same model.

After the backward pass:

**4 local gradients → collective reduction → synchronized gradient → same update on every replica**

If communication becomes too expensive, adding more GPUs can give very little extra speedup.


## 15 - Multi-GPU and multi-node launch patterns

### PyTorch: one node, four GPUs

```bash
torchrun --standalone --nproc_per_node=4 train_ddp.py
```

### PyTorch: two nodes, four GPUs per node

```bash
torchrun --nnodes=2 --nproc_per_node=4 \
         --node_rank=<0-or-1> \
         --master_addr=<host> --master_port=29500 \
         train_ddp.py
```

This gives **8 training processes**, normally one process per GPU.

### TensorFlow: one host

```python
strategy = tf.distribute.MirroredStrategy()
```

### TensorFlow: several workers

```python
strategy = tf.distribute.MultiWorkerMirroredStrategy()
```

Example cluster description for two TensorFlow workers:

```json
{
  "cluster": {
    "worker": ["nodeA:12345", "nodeB:12345"]
  },
  "task": {
    "type": "worker",
    "index": 0
  }
}
```


In [ ]:
nodes=2
gpus_per_node=4
workers=nodes*gpus_per_node
global_batch=1024
local_batch=global_batch//workers

print("nodes:", nodes)
print("GPUs per node:", gpus_per_node)
print("training workers:", workers)
print("global batch:", global_batch)
print("local batch per worker:", local_batch)
print("\nNew cost to watch: gradient communication across the interconnect.")


## 16 - What resources does an ML job need?

A GPU training job usually needs more than a GPU.

- **GPU:** forward and backward computation
- **CPU cores:** loading, decoding and preprocessing
- **memory:** batches and worker processes
- **storage bandwidth:** datasets and checkpoints

A typical Slurm request could include:

```bash
#SBATCH --job-name=ml-demo
#SBATCH --gpus=1
#SBATCH --cpus-per-task=8
#SBATCH --mem=32G
```

The exact partition and GPU options depend on the cluster configuration.


In [ ]:
if shutil.which("sinfo"):
    print("Visible Slurm partitions and generic resources:")
    print(sh("sinfo -o '%P  GRES=%G  CPUs=%c  MemMB=%m' | head -15"))
else:
    print("sinfo is not visible in this environment.")

print("\nQuestion: why request CPU cores for a GPU training job?")
print("Expected: loading, decoding, preprocessing, augmentation and orchestration can run on CPUs.")


# 17 - Exit ticket

A training job shows:

- GPU utilization: **35%**
- GPU memory: **45%**
- CPU utilization: **100%**
- continuous storage reads

**Would adding more GPUs be your first change? Why or why not?**

<details>
<summary>Check your answer</summary>

Probably not. The evidence suggests that CPU preprocessing or the data pipeline is the main limitation. Extra GPUs may spend more time waiting for input.

</details>

### Remember

**pipeline → bottleneck → evidence → change one factor → measure again**

### Next

Multiple GPUs add compute capacity, but they also add **communication and synchronization**.
